# 🔭 Notebook: Observability with Langfuse (Self-Hosted) — *Part 3 of 3*

*⚠️ Advanced and optional, like `10_2_langsmith.ipynb` - not required for the core course. This notebook additionally needs [Docker](https://www.docker.com/products/docker-desktop/) installed and running.*

*This is the last of three notebooks in this chapter: 1) `10_1_pytest.ipynb` (general testing) → 2) LangSmith (hosted) → 3) **Langfuse** (self-hosted). Same ideas — traces, runs, metadata, feedback — on a tool you run yourself instead of a third party's cloud.*

## 📚 Sources

- [Langfuse: Docker Compose self-hosting guide](https://langfuse.com/self-hosting/deployment/docker-compose)
- [Langfuse: LangChain/LangGraph integration](https://langfuse.com/docs/integrations/langchain/tracing)

## LangSmith vs. Langfuse: why bother with a second tool?

`10_2_langsmith.ipynb` covered the *concepts* of observability - traces, runs, metadata, feedback - using LangSmith, a hosted service. Everything you learned there transfers directly to Langfuse; the vocabulary differs slightly (a "run" is called an "observation," "feedback" is called a "score"), but the ideas are identical.

The reason to also see Langfuse: **LangSmith's self-hosting option is an Enterprise-only feature** - you can't run it on your own infrastructure without an Enterprise license. If you ever need traces to never leave your own servers (a hard compliance requirement, an air-gapped environment, or simply not wanting a third party to see your prompts), you need a genuinely self-hostable alternative. [Langfuse](https://langfuse.com/) is open-source and one of the most popular such alternatives - and unlike LangSmith, you can run the whole stack yourself with Docker Compose, for free, right now.

## Setting up Langfuse with Docker Compose

Langfuse isn't one program - it's several services working together: a Next.js web app, a background worker, and three datastores (Postgres for app data, ClickHouse for trace data, Redis for queues/caching, plus MinIO for file storage). Docker Compose runs all of them together from one file.

### Step 1: Get the `docker-compose.yml`

We've already placed the official, unmodified file in `langfuse/docker-compose.yml` in this chapter folder (from [Langfuse's GitHub repo](https://github.com/langfuse/langfuse/blob/main/docker-compose.yml)). If you ever need a fresh copy:

```bash
curl -fsSL https://raw.githubusercontent.com/langfuse/langfuse/main/docker-compose.yml -o langfuse/docker-compose.yml
```

<details>
<summary><b>Port already in use?</b></summary>

If `docker compose up` fails with something like `address already in use`, another program on your machine already has that port. Find out what with `lsof -nP -iTCP:<port> -sTCP:LISTEN` (macOS/Linux) and either stop that program, or change the *host* side of that port mapping in `docker-compose.yml` (the number before the colon - the container-internal side, after the colon, must stay as-is).

</details>

### Step 2: Configure headless initialization

By default, starting Langfuse gives you an empty instance - you'd sign up through the web UI, create an organization and project by hand, then copy API keys out of the settings page. For a course notebook, there's a smoother way: Langfuse supports **headless initialization** - a set of `LANGFUSE_INIT_*` environment variables that auto-create an org, a project, a user, and even a specific API key pair, the moment the containers start for the first time.

Create `langfuse/.env` (copy `langfuse/.env.example` and fill it in) with:

```
ENCRYPTION_KEY=<run: openssl rand -hex 32>
LANGFUSE_INIT_ORG_ID=ai-engineering-course
LANGFUSE_INIT_ORG_NAME=AI Engineering Course
LANGFUSE_INIT_PROJECT_ID=ai-engineering-course
LANGFUSE_INIT_PROJECT_NAME=ai-engineering-course
LANGFUSE_INIT_PROJECT_PUBLIC_KEY=pk-lf-<any random string>
LANGFUSE_INIT_PROJECT_SECRET_KEY=sk-lf-<any random string>
LANGFUSE_INIT_USER_EMAIL=student@example.com
LANGFUSE_INIT_USER_NAME=Student
LANGFUSE_INIT_USER_PASSWORD=<pick a password>
```

This `.env` lives *inside* `langfuse/` and is read automatically by `docker compose` (it looks for a `.env` file in the same directory as the `docker-compose.yml`) - it's separate from the project-root `.env` every other notebook in this course reads via `load_dotenv()`.

### Step 3: Start the containers

```bash
cd langfuse
docker compose up -d
```

`-d` runs the containers in the background so your terminal stays free. The first run pulls several images (ClickHouse and the Langfuse images are each around 1 GB), so this can take a few minutes depending on your connection. Check on it with:

```bash
docker compose ps
```

Once every service shows `Up ... (healthy)`, Langfuse is ready at **[http://localhost:3000](http://localhost:3000)** - open it in a browser and log in with the `LANGFUSE_INIT_USER_EMAIL` / `LANGFUSE_INIT_USER_PASSWORD` you chose, to see the actual UI (traces will show up here as we produce them below).

When you're done with this notebook, free up the resources with `docker compose down` (from inside `langfuse/`) - or add `-v` to also delete the stored trace data.

### Step 4: Point this notebook at your local instance

The Python side reads different environment variable names than the Docker Compose init step does - copy the *same* public/secret key values you put in `langfuse/.env` into the project-root `.env` (the one `LLM_HOST` already lives in), under these names:

```
LANGFUSE_PUBLIC_KEY=pk-lf-<the same value as LANGFUSE_INIT_PROJECT_PUBLIC_KEY>
LANGFUSE_SECRET_KEY=sk-lf-<the same value as LANGFUSE_INIT_PROJECT_SECRET_KEY>
LANGFUSE_HOST=http://localhost:3000
```

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads LLM_HOST, LANGFUSE_PUBLIC_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_HOST from .env

LLM_HOST = os.environ["LLM_HOST"]
LLM_URL = f"http://{LLM_HOST}:11434"
LLM_REASONING = "gemma4:26b"

In [2]:
from langfuse import get_client

lf = get_client()
print("Connected to local Langfuse instance:", lf.auth_check())

Connected to local Langfuse instance: True


## Automatic tracing for LangChain and LangGraph

Langfuse integrates with LangChain via a **callback handler** rather than an environment-variable switch - pass `CallbackHandler()` in `config={"callbacks": [...]}` on any `.invoke()` call. Everything else about the agent is unchanged from `06_2_agents.ipynb`.

In [3]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langchain.tools import tool
from langfuse.langchain import CallbackHandler

llm = ChatOllama(model=LLM_REASONING, base_url=LLM_URL, temperature=0)


@tool
def get_temperature(city: str) -> str:
    """Get the current temperature for a city."""
    temperatures = {"New York": "22°C", "London": "15°C", "Tokyo": "18°C"}
    return temperatures.get(city, "Unknown")


weather_agent = create_agent(model=llm, tools=[get_temperature])
langfuse_handler = CallbackHandler()

result = weather_agent.invoke(
    {"messages": [{"role": "user", "content": "What is the temperature in Tokyo?"}]},
    config={"callbacks": [langfuse_handler]},
)
print(result["messages"][-1].content)

The current temperature in Tokyo is 18°C.


## Verifying the trace

Same idea as `10_2`: instead of (or in addition to) opening [localhost:3000](http://localhost:3000) in a browser, we can pull the trace straight back through the API and inspect its nested structure from code.

In [4]:
import time

time.sleep(2)  # give the background worker a moment to ingest the trace
lf.flush()  # make sure any buffered events are sent before we query

# LangGraph names its root trace "LangGraph" - filter by name rather than
# just taking the most recent trace, since this project accumulates traces
# from every run of this notebook (yours and any earlier ones)
latest_trace = lf.api.trace.list(limit=1, name="LangGraph").data[0]
full_trace = lf.api.trace.get(latest_trace.id)

print("Observations inside the trace:")
for obs in full_trace.observations:
    print(f"  {obs.name:16s} [{obs.type}]")

Observations inside the trace:
  LangGraph        [CHAIN]
  ChatOllama       [GENERATION]
  model            [CHAIN]
  tools            [CHAIN]
  get_temperature  [TOOL]
  model            [CHAIN]
  ChatOllama       [GENERATION]


`LangGraph` is the trace root (the whole `.invoke()` call), containing `ChatOllama` generations, `tools`/`model` chain steps, and the `get_temperature` tool call - the entire internal graph structure `create_agent` builds, captured with the one callback handler line above.

## Manual instrumentation with `@observe`

Langfuse's equivalent of LangSmith's `@traceable` is `@observe()` - decorate any function, and its call becomes a span (or, with `as_type="generation"`, specifically an LLM-call span). Same "library help desk" example as `10_2`, for a direct comparison between the two tools.

In [5]:
import openai
from langfuse import observe

client = openai.OpenAI(base_url=f"{LLM_URL}/v1", api_key="ollama")

library_faqs = [
    "Books can be borrowed for 21 days and renewed once online.",
    "Overdue books incur a fine of €0.20 per day.",
    "The library is open Monday-Saturday, 8:00-22:00.",
]


@observe()
def get_context(question: str) -> str:
    return "\n".join(library_faqs)


@observe(as_type="generation")
def call_llm(question: str, context: str) -> str:
    response = client.chat.completions.create(
        model=LLM_REASONING,
        messages=[
            {"role": "system", "content": f"Answer using only the context below.\n\nContext:\n{context}"},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content


@observe()
def assistant(question: str) -> str:
    context = get_context(question)
    return call_llm(question, context)


answer = assistant("How long can I borrow a book for, and what happens if I'm late?")
print(answer)

lf.flush()
time.sleep(2)
latest = lf.api.trace.list(limit=1, name="assistant").data[0]
full = lf.api.trace.get(latest.id)
print("\nTrace tree:")
for obs in full.observations:
    print(f"  {obs.name:12s} [{obs.type}]")

Books can be borrowed for 21 days, and overdue books incur a fine of €0.20 per day.

Trace tree:
  call_llm     [GENERATION]
  assistant    [SPAN]
  get_context  [SPAN]


Same three-level structure as LangSmith's version in `10_2`: `assistant` (span) → `get_context` (span) and `call_llm` (generation) - the concepts really are identical, only the decorator name and the vocabulary ("observation" instead of "run") differ.

## Metadata

Inside an `@observe`-decorated function, `lf.update_current_span(metadata={...})` attaches arbitrary key-value pairs to that specific observation - the same filtering/grouping use case as LangSmith's `metadata=` in `10_2`.

In [6]:
@observe()
def tagged_assistant(question: str) -> str:
    lf.update_current_span(metadata={"model": LLM_REASONING, "course_chapter": "10"})
    response = client.chat.completions.create(
        model=LLM_REASONING,
        messages=[{"role": "user", "content": question}],
    )
    return response.choices[0].message.content


tagged_assistant("Say hello in one short sentence.")
lf.flush()
time.sleep(2)

latest = lf.api.trace.list(limit=1, name="tagged_assistant").data[0]
custom_metadata = {k: v for k, v in latest.metadata.items() if k in ("model", "course_chapter")}
print("Custom metadata on the trace:", custom_metadata)

Custom metadata on the trace: {'model': 'gemma4:26b', 'course_chapter': 10}


## Scores (Langfuse\'s feedback)

Langfuse calls the same idea - attaching a score to a specific trace - a **score** instead of "feedback." Capture the trace ID with `lf.get_current_trace_id()` inside the observed function, then call `create_score` afterward.

In [7]:
captured_trace_id = {}


@observe()
def scored_assistant(question: str) -> str:
    captured_trace_id["id"] = lf.get_current_trace_id()
    return "The library is open Monday-Saturday, 8:00-22:00."


answer = scored_assistant("What are the library's opening hours?")
print(answer)

lf.flush()
time.sleep(1)

trace_id = captured_trace_id["id"]
lf.create_score(trace_id=trace_id, name="user-score", value=1.0, comment="Correct and concise.")
lf.flush()

# Score ingestion can lag a couple of seconds behind create_score() returning, so retry briefly
scores = []
for _ in range(5):
    time.sleep(2)
    scores = lf.api.trace.get(trace_id).scores
    if scores:
        break

print("Scores on this trace:", [(s.name, s.value, s.comment) for s in scores])

The library is open Monday-Saturday, 8:00-22:00.
Scores on this trace: [('user-score', 1.0, 'Correct and concise.')]


## Exercise: Trace and tag a RAG-style call

Same exercise as `10_2_langsmith.ipynb`, now with Langfuse. Write an `@observe()`-decorated function `search_and_answer(question)` that:
1. Calls an `@observe()` helper that returns 2-3 fixed strings of your choosing (any topic).
2. Passes them as context into an LLM call via `@observe(as_type="generation")`.
3. Inside `search_and_answer`, calls `lf.update_current_span(metadata={"exercise": "10_3"})`.

Then use `lf.api.trace.get(...)` to confirm: the helper shows up as a nested observation, and the metadata is attached.

In [8]:
# Insert code here...

<details>
<summary><b>Show solution</b></summary>

```python
@observe()
def retrieve(query: str) -> list[str]:
    return [
        "The Eiffel Tower was completed in 1889.",
        "It was originally intended as a temporary structure.",
    ]


@observe(as_type="generation")
def generate(question: str, context: list[str]) -> str:
    response = client.chat.completions.create(
        model=LLM_REASONING,
        messages=[
            {"role": "system", "content": f"Answer using only this context:\n{chr(10).join(context)}"},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content


@observe()
def search_and_answer(question: str) -> str:
    lf.update_current_span(metadata={"exercise": "10_3"})
    context = retrieve(question)
    return generate(question, context)


search_and_answer("When was the Eiffel Tower completed?")
lf.flush()
time.sleep(2)

latest = lf.api.trace.list(limit=1, name="search_and_answer").data[0]
full = lf.api.trace.get(latest.id)
print("Observations:", [(o.name, o.type) for o in full.observations])
print("Metadata:", full.metadata.get("exercise"))
```

</details>

---

Don't forget to `docker compose down` (from inside `langfuse/`) once you're done experimenting, to free up the resources.